In [1]:
import os
os.environ["FLAGS_use_mkldnn"] = "0"

In [2]:
import sys
sys.path.append("../src")
from parser import run_extraction, EXTRACTORS, sanitize_filename
import os
import shutil

c:\Users\USER\OneDrive\Documents\ESSAI-assistant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
normal_folder = "../data/raw/normal_pdfs"
all_results = []

for filename in os.listdir(normal_folder):
    if not filename.endswith(".pdf"):
        continue
    path = os.path.join(normal_folder, filename)

    for extractor_name in ["pdfplumber", "pymupdf"]:
        try:
            data, output_path = run_extraction(path, extractor_name)
            all_results.append({
                "filename": filename,
                "extractor": extractor_name,
                "char_count": sum(p["char_count"] for p in data["pages"]),
                "num_tables": sum(len(p.get("tables", [])) for p in data["pages"]),
                "output_path": output_path
            })
            print(f"✅ {filename} → {extractor_name}")
        except Exception as e:
            print(f"❌ Erreur sur {filename} ({extractor_name}) : {e}")

✅ association_clubs.pdf → pdfplumber
✅ association_clubs.pdf → pymupdf
✅ Ateliers_statistique.pdf → pdfplumber
✅ Ateliers_statistique.pdf → pymupdf
✅ Charte_des_clubs.pdf → pdfplumber
✅ Charte_des_clubs.pdf → pymupdf
✅ Convention_PFE_ESSAI.pdf → pdfplumber
✅ Convention_PFE_ESSAI.pdf → pymupdf
✅ FAQ_ESSAI.pdf → pdfplumber
✅ FAQ_ESSAI.pdf → pymupdf
✅ Guide_Etudiant.pdf → pdfplumber
✅ Guide_Etudiant.pdf → pymupdf
✅ Modalités_Admission_DoubleDiplome.pdf → pdfplumber
✅ Modalités_Admission_DoubleDiplome.pdf → pymupdf
✅ Note_PFE.pdf → pdfplumber
✅ Note_PFE.pdf → pymupdf
✅ organisation_activitésclubs_ESSAI.pdf → pdfplumber
✅ organisation_activitésclubs_ESSAI.pdf → pymupdf
✅ Regime_etudes_ESSAI2022.pdf → pdfplumber
✅ Regime_etudes_ESSAI2022.pdf → pymupdf
✅ Règlement_Bib.pdf → pdfplumber
✅ Règlement_Bib.pdf → pymupdf
✅ Règlement_Examen_ESSAI.pdf → pdfplumber
✅ Règlement_Examen_ESSAI.pdf → pymupdf
✅ Règlement_intérieur.pdf → pdfplumber
✅ Règlement_intérieur.pdf → pymupdf


In [4]:
scanned_folder = "../data/raw/scanned"

for filename in os.listdir(scanned_folder):
    if not filename.endswith(".pdf"):
        continue

    clean_name = sanitize_filename(filename)
    original_path = os.path.join(scanned_folder, filename)

    if clean_name != filename:
        clean_path = os.path.join(scanned_folder, clean_name)
        if not os.path.exists(clean_path):
            shutil.copy(original_path, clean_path)
        path = clean_path
        print(f"ℹ️ Renommé pour compatibilité : {filename} → {clean_name}")
    else:
        path = original_path

    try:
        data, output_path = run_extraction(path, "docling")
        char_count = sum(p["char_count"] for p in data["pages"])
        num_tables = sum(len(p.get("tables", [])) for p in data["pages"])

        all_results.append({
            "filename": clean_name,
            "extractor": "docling",
            "char_count": char_count,
            "num_tables": num_tables,
            "output_path": output_path
        })
        print(f"✅ {clean_name} → docling ({char_count} caractères, {num_tables} tableau(x))")
    except Exception as e:
        print(f"❌ Erreur sur {clean_name} : {e}")

[INFO] 2026-08-03 15:13:41,582 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 15:13:41,649 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\USER\OneDrive\Documents\ESSAI-assistant\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 15:13:41,650 [RapidOCR] main.py:63: Using C:\Users\USER\OneDrive\Documents\ESSAI-assistant\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-03 15:13:41,724 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-03 15:13:41,740 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\USER\OneDrive\Documents\ESSAI-assistant\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 15:13:41,740 [RapidOCR] main.py:63: Using C:\Users\USER\OneDrive\Documents\ESSAI-assistant\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-03 15:13:41,787 [RapidOCR] base.py:23: Usin

✅ communique_d_inscription_pdf.pdf → docling (2883 caractères, 1 tableau(x))


In [5]:
import json

metadata_path = "../data/processed/metadata/extraction_results.json"
os.makedirs(os.path.dirname(metadata_path), exist_ok=True)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n{len(all_results)} extractions enregistrées dans {metadata_path}")


27 extractions enregistrées dans ../data/processed/metadata/extraction_results.json
